> **Código académico, sin garantía.** Material de los cursos de Modelado y Diseño de Líneas de Transmisión (Universidad Nacional de Colombia). Se entrega tal cual, sin garantía de ninguna clase; **no debe usarse para decisiones de diseño, operación o seguridad de instalaciones reales** sin verificación independiente. Ver [`DISCLAIMER.md`](../../DISCLAIMER.md). Licencia por definir (intención: código abierto).

# Cálculo de la corriente máxima por conductor desnudo

Basado en la norma **IEEE 738-2012**  
Usa la **aproximación estática**  
Se aproxima a **condiciones del Ecuador**, para **radiación máxima**  
Valores de referencia de la guía del curso (Cardona Correa, UNAL Sede Medellín, 2022)


In [ ]:
import numpy as np


Unidades: todas las magnitudes están en SI (m, °C, W/m, Ω/m, W/m²).

### Definición de Parámetros

In [ ]:
d=0.01596       # Diámetro del conductor [m]
E=.5            # Emisividad del conductor (0.23 a 0.91; 0.5 para superficie envejecida)
He=0.           # Altura sobre el nivel del mar [m]
Vw=0.610        # Velocidad del viento [m/s]
Rac1=0.189E-3   # Resistencia AC del conductor a T1 [Ohm/m]
Rac2=0.231e-3   # Resistencia AC del conductor a T2 [Ohm/m]
T1=25           # Temperatura a la que se mide Rac1 [°C]
T2=75           # Temperatura a la que se mide Rac2 [°C]
Ta=25           # Temperatura ambiente [°C]; sin dato, la guía sugiere Ta=(6150-He)/149
Tc=75           # Temperatura máxima del conductor [°C]
alfa=0.5        # Constante de absorción solar del conductor (0.23 a 0.91; 0.5 para superficie envejecida)
phi=np.radians(90) # Ángulo entre la dirección del viento y el eje del conductor [rad] (90° = viento perpendicular)
Qse=1043        # Radiación solar corregida por altura [W/m^2] (valor UPME citado en la guía del curso)


Temperatura promedio frontera conductor (Temperatura ambiente y superficial del conductor)

In [ ]:
Tfilm=(Tc+Ta)/2
print('Temperatura de película Tfilm [°C]:', Tfilm)

Cálculo de la densidad del aire

In [ ]:

pf=(1.293-1.525E-4*He+6.379e-9*He**2)/(1+0.00367*Tfilm) #Densidad del aire [kg/m^3] 
print('Densidad del aire [kg/m^3] :', pf)

Viscosidad Dinámica del aire

In [ ]:
mu_f=(1.458e-6*(Tfilm+273.15)**1.5)/(Tfilm+273.15+110.4)
print('Viscosidad dinámica del aire [kg/(m·s)] :', mu_f)


Conductividad Térmica

In [ ]:
kf=2.424e-2+7.477e-5*Tfilm-4.407e-9*Tfilm**2 
print('Conductividad térmica del aire [W/(m·K)] :', kf)

Cálculo de la resistencia del conductor a la temperatura de trabajo Tc.

In [ ]:
Rac=Rac1+(Tc-T1)*(Rac2-Rac1)/(T2-T1) # Resistencia AC del conductor a Tc [Ohm/m]
print('Resistencia AC del conductor a Tc [Ohm/m] :', Rac)


In [ ]:
kangle=1.194-np.cos(phi)+0.194*np.cos(2*phi)+0.368*np.sin(2*phi)
print('Factor de dirección del viento kangle:',kangle)


In [ ]:
NRE=d*pf*Vw/mu_f  # Número de Reynolds [adimensional] (d=diámetro [m], pf=densidad del aire [kg/m^3], Vw=viento [m/s], mu_f=viscosidad dinámica [kg/(m·s)])
print('Número de Reynolds:', NRE)


Cálculo de la tasa de transferencia de calor por convección

In [ ]:
qcn=3.645*pf**0.5*d**0.75*(Tc-Ta)**1.25 #Tasa de transferencia de calor por convección natural [W/m] (D=diámetro del conductor en metros, Tc=Temperatura del conductor en °C, Ta=Temperatura del aire en °C)
print('Tasa de transferencia de calor por convección natural [W/m] :', qcn)
qc1=(1.01+1.35*(NRE)**0.52)*kf*kangle*(Tc-Ta) #Tasa de transferencia de calor por convección forzada [W/m] (Vw=velocidad del viento en m/s, mu_f=viscosidad dinámica del aire en kg/(m·s), D=diámetro del conductor en metros, Tc=Temperatura del conductor en °C, Ta=Temperatura del aire en °C)
print('Tasa de transferencia de calor por convección forzada [W/m] :', qc1)
qc2=(0.754*(NRE)**0.6)*kf*kangle*(Tc-Ta) #Tasa de transferencia de calor por convección forzada [W/m] (Vw=velocidad del viento en m/s, mu_f=viscosidad dinámica del aire en kg/(m·s), D=diámetro del conductor en metros, Tc=Temperatura del conductor en °C, Ta=Temperatura del aire en °C)
print('Tasa de transferencia de calor por convección forzada [W/m] :', qc2)

In [ ]:
qc=max(qcn, qc1, qc2)  # Pérdida de calor por convección [W/m]
print(f'Convección total qc: {qc:.2f} W/m')


Tasa de transferencia de calor por radiación

In [ ]:
qr=17.8*E*d*(((Tc+273.15)/100)**4-((Ta+273.15)/100)**4)  # Pérdida de calor por radiación [W/m] (E=emisividad, Tc y Ta en °C, d=diámetro [m])
print(f'Radiación qr: {qr:.2f} W/m')


Ganancia de calor por radiación solar

In [ ]:
qs=alfa*Qse*d  # Ganancia de calor solar [W/m] (alfa=constante de absorción solar, Qse=radiación solar [W/m^2], d=diámetro [m]); se asume sol perpendicular, sen(theta)=1 (forma simplificada para la zona tropical)
print(f'Radiación solar qs: {qs:.2f} W/m')


Cálculo de la corriente del conductor para las condiciones establecidas

In [ ]:

Icond=np.sqrt((qc + qr - qs)/ Rac)  
print('(Corriente máxima del conductor) [A] :', Icond)